<a href="https://colab.research.google.com/github/ablasve/Mini-Proyecto-Asistente-Multimodal-de-Salud/blob/main/Prueba_asistente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PRUEBA BUCLE FINAL

## 0. Instalaciones e importaciones

In [1]:
!pip install -q bitsandbytes openai-whisper edge-tts transformers accelerate qwen-vl-utils pdf2image
!apt-get install -y poppler-utils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 45 not upgraded.


In [2]:
import os
from google.colab import files

nombre_archivo = "funciones_salud.py"

# 1. Comprobamos si ya existe una versión anterior y la borramos
if os.path.exists(nombre_archivo):
    os.remove(nombre_archivo)
    print(f"🗑️ Versión anterior de '{nombre_archivo}' eliminada con éxito.")

# 2. Pedimos que subas la nueva versión
print(f"⬆️ Por favor, sube el nuevo archivo {nombre_archivo}:")
uploaded = files.upload()

# Confirmación
if nombre_archivo in uploaded:
    print(f"✅ Archivo '{nombre_archivo}' subido y listo para importar.")

🗑️ Versión anterior de 'funciones_salud.py' eliminada con éxito.
⬆️ Por favor, sube el nuevo archivo funciones_salud.py:


Saving funciones_salud.py to funciones_salud.py
✅ Archivo 'funciones_salud.py' subido y listo para importar.


In [3]:
import torch
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer, Qwen2VLForConditionalGeneration, AutoProcessor
import whisper

#obligar a recargar el .py si detectan cambios
import importlib
import funciones_salud

def reload_funciones():
    importlib.reload(funciones_salud)
    return funciones_salud

funciones = reload_funciones()

In [4]:
import os
from google.colab import userdata

# Cogemos el secreto de la llave 🔑 y se lo damos al sistema
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

print("✅ Token de Hugging Face configurado correctamente.")

✅ Token de Hugging Face configurado correctamente.


## 1. Carga de modelos

In [5]:
# Cuantización para ahorrar memoria en Colab
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float32
)

print("Cargando Whisper (Audio a Texto)...")
model_whisper = whisper.load_model("small")

print("Cargando Qwen Texto (Lógica y Extracción)...")
id_model_texto = "Qwen/Qwen2.5-3B-Instruct"
tokenizer_texto = AutoTokenizer.from_pretrained(id_model_texto)
model_texto = AutoModelForCausalLM.from_pretrained(
    id_model_texto, quantization_config=bnb_config, device_map="auto"
)

print("Cargando Qwen Visión (Lectura de imágenes/PDF)...")
id_model_vision = "Qwen/Qwen2-VL-2B-Instruct"
processor_vision = AutoProcessor.from_pretrained(id_model_vision)
model_vision = Qwen2VLForConditionalGeneration.from_pretrained(
    id_model_vision, quantization_config=bnb_config, device_map="auto"
)

print("✅ ¡Todos los modelos cargados y listos!")

Cargando Whisper (Audio a Texto)...
Cargando Qwen Texto (Lógica y Extracción)...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Cargando Qwen Visión (Lectura de imágenes/PDF)...


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

✅ ¡Todos los modelos cargados y listos!


## 2. Llamada al asistente

Cuando el código de Python le dice al navegador: "Oye, enciende el micro durante 5 segundos", el navegador detiene todo el reloj interno y lanza el mensaje emergente de "¿Permitir usar el micrófono?".
Mientras nosotros tardamos 2 o 3 segundos en leerlo y darle a "Permitir", Python y el navegador se desincronizan. El código se queda esperando una respuesta que ya ha caducado, y el bucle colapsa. Al reiniciar el kernel, el navegador ya tiene el permiso guardado, no saca el mensaje emergente, y todo funciona correctamente.

Para arreglar esto, vamos a añadir una celda de calentamiento del micrófono justo antes de la celda donde llamamos al asistente:

In [7]:
# ==========================================
# CELDA DE CALENTAMIENTO DEL MICRÓFONO
# ==========================================
print("⚠️ Atento al navegador: Acepta los permisos del micrófono si te lo pide.")
print("Haciendo una prueba de audio de 1 segundo...")

# Hacemos una grabación "fantasma" muy cortita solo para forzar el pop-up
funciones.grabar_audio(segundos=1)

print("✅ ¡Micrófono listo y configurado! Ya puedes ejecutar la celda de abajo.")

⚠️ Atento al navegador: Acepta los permisos del micrófono si te lo pide.
Haciendo una prueba de audio de 1 segundo...


✅ ¡Micrófono listo y configurado! Ya puedes ejecutar la celda de abajo.


Ahora ya podemos llamar al asistente con los permisos del micrófono aceptados:

In [8]:
# Llamamos al bucle principal pasándole todos los modelos que acabamos de cargar
await funciones.iniciar_asistente(model_whisper, model_texto, tokenizer_texto, model_vision, processor_vision)

CancelledError: 